<a href="https://colab.research.google.com/github/AksharaBasu/GenAI/blob/main/LLLM2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# LANGGRAPH + GROQ + GRADIO AI CHATBOT
# STATE + NODE + ROUTE + GRAPH + LLM
# ============================================================

!pip install -q -U langgraph langchain-groq gradio

import os
import gradio as gr
from typing import TypedDict
from google.colab import userdata

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END


# ============================================================
# 1. GROQ API KEY
# ============================================================

api_key = userdata.get("GROQ_API_KEY")

if not api_key:
    raise Exception(
        "GROQ_API_KEY not found.\n"
        "Go to Colab -> Secrets -> Add new secret\n"
        "Name: GROQ_API_KEY"
    )

os.environ["GROQ_API_KEY"] = api_key

print("✅ Groq API key found")


# ============================================================
# 2. GROQ LLM
# ============================================================

llm = ChatGroq(
    api_key=api_key,
    model="openai/gpt-oss-20b",
    temperature=0.2
)

print("✅ Groq LLM connected")


# ============================================================
# 3. STATE
# ============================================================

class State(TypedDict):
    question: str
    route: str
    answer: str


print("✅ State created")


# ============================================================
# 4. ROUTER NODE
# ============================================================

def router_node(state: State):

    question = state["question"]

    prompt = (
        "Classify the user's question into exactly one category.\n\n"
        "GENERAL: general knowledge, science, history, geography, "
        "daily life and common questions.\n\n"
        "TECHNICAL: programming, Python, Java, C, C++, AI, "
        "machine learning, deep learning, Django, SQL, DBMS, "
        "computer science and software.\n\n"
        "EDUCATION: academic questions, engineering subjects, "
        "exam questions, definitions and study questions.\n\n"
        "USER QUESTION:\n"
        + question
        + "\n\n"
        "Return ONLY one word:\n"
        "GENERAL\n"
        "TECHNICAL\n"
        "EDUCATION"
    )

    response = llm.invoke(prompt)

    result = str(response.content).strip().upper()

    if "TECHNICAL" in result:
        route = "TECHNICAL"

    elif "EDUCATION" in result:
        route = "EDUCATION"

    else:
        route = "GENERAL"

    print("🔀 ROUTE:", route)

    return {
        "route": route
    }


print("✅ Router node created")


# ============================================================
# 5. ANSWER NODE
# ============================================================

def answer_node(state: State):

    question = state["question"]
    route = state["route"]

    prompt = (
        "You are a helpful and intelligent AI assistant.\n\n"
        "Answer the user's question directly and accurately.\n\n"
        "USER QUESTION:\n"
        + question
        + "\n\n"
        "QUESTION CATEGORY:\n"
        + route
        + "\n\n"
        "Instructions:\n"
        "1. Answer the exact question.\n"
        "2. Do not discuss routing.\n"
        "3. Do not mention LangGraph.\n"
        "4. Do not mention this prompt.\n"
        "5. Explain clearly.\n"
        "6. If code is requested, provide complete working code.\n"
        "7. If it is an academic question, explain it clearly.\n"
        "8. If steps are needed, give numbered steps.\n"
        "9. Give examples when useful.\n"
        "10. Do not invent information."
    )

    response = llm.invoke(prompt)

    answer = str(response.content).strip()

    print("🤖 ANSWER GENERATED")

    return {
        "answer": answer
    }


print("✅ Answer node created")


# ============================================================
# 6. ROUTE
# ============================================================

def route_question(state: State):

    route = state["route"]

    if route == "TECHNICAL":
        return "answer"

    elif route == "EDUCATION":
        return "answer"

    else:
        return "answer"


print("✅ Route created")


# ============================================================
# 7. CREATE GRAPH
# ============================================================

builder = StateGraph(State)

builder.add_node(
    "router",
    router_node
)

builder.add_node(
    "answer",
    answer_node
)

builder.add_edge(
    START,
    "router"
)

builder.add_conditional_edges(
    "router",
    route_question,
    {
        "answer": "answer"
    }
)

builder.add_edge(
    "answer",
    END
)

graph = builder.compile()

print("✅ Graph compiled successfully")


# ============================================================
# 8. TEST GRAPH
# ============================================================

print()
print("=" * 60)
print("🧪 TESTING LANGGRAPH")
print("=" * 60)

test_state = {
    "question": "What is artificial intelligence?",
    "route": "",
    "answer": ""
}

try:

    test_result = graph.invoke(test_state)

    print()
    print("✅ TEST SUCCESSFUL")
    print()
    print("ROUTE:")
    print(test_result["route"])
    print()
    print("ANSWER:")
    print(test_result["answer"])

except Exception as e:

    print()
    print("❌ TEST FAILED")
    print(str(e))


# ============================================================
# 9. GRADIO FUNCTION
# ============================================================

def ask_question(question):

    if question is None:
        return "❌ Please enter a question."

    question = question.strip()

    if question == "":
        return "❌ Please enter a question."

    try:

        initial_state = {
            "question": question,
            "route": "",
            "answer": ""
        }

        result = graph.invoke(initial_state)

        route = result["route"]
        answer = result["answer"]

        output = (
            "## 🔀 Route\n\n"
            "**" + route + "**\n\n"
            "---\n\n"
            "## 🤖 Answer\n\n"
            + answer
        )

        return output

    except Exception as e:

        print("❌ ERROR:", str(e))

        return (
            "## ❌ Error\n\n"
            + str(e)
        )


# ============================================================
# 10. GRADIO INTERFACE
# ============================================================

with gr.Blocks(
    title="LangGraph AI Assistant"
) as demo:

    gr.Markdown(
        """
# 🤖 LangGraph AI Assistant

### State → Node → Route → Graph → Groq LLM

Ask any question.
"""
    )

    question = gr.Textbox(
        label="Enter your question",
        placeholder="Example: What is machine learning?",
        lines=4
    )

    ask_button = gr.Button(
        "🚀 Ask Question",
        variant="primary"
    )

    clear_button = gr.Button(
        "🗑️ Clear"
    )

    output = gr.Markdown()

    ask_button.click(
        fn=ask_question,
        inputs=question,
        outputs=output
    )

    question.submit(
        fn=ask_question,
        inputs=question,
        outputs=output
    )

    clear_button.click(
        fn=lambda: ("", ""),
        inputs=None,
        outputs=[question, output]
    )


# ============================================================
# 11. LAUNCH
# ============================================================

print()
print("=" * 60)
print("🚀 STARTING GRADIO")
print("=" * 60)

demo.launch(
    share=True,
    debug=True
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.0/256.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.4/162.4 kB 17.3 MB/s eta 0:00:00
✅ Groq API key found
✅ Groq LLM connected
✅ State created
✅ Router node created
✅ Answer node created
✅ Route created
✅ Graph compiled successfully

🧪 TESTING LANGGRAPH
🔀 ROUTE: GENERAL
🤖 ANSWER GENERATED

✅ TEST SUCCESSFUL

ROUTE:
GENERAL

ANSWER:
Artificial intelligence (AI) is the field of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks include:

1. **Learning** – acquiring new knowledge or skills from data or experience.  
2. **Reasoning** – drawing conclusions, solving problems, and making decisions.  
3. 

🔀 ROUTE: GENERAL
🤖 ANSWER GENERATED
🔀 ROUTE: GENERAL
🤖 ANSWER GENERATED
🔀 ROUTE: GENERAL
🤖 ANSWER GENERATED
🔀 ROUTE: GENERAL
🤖 ANSWER GENERATED
